# Numba: JIT Compilation for Numerical Python

This notebook is a **from-scratch to intermediate/advanced** guide to **Numba**: just-in-time
compilation of Python code to fast machine code using the LLVM toolchain.

We cover:

1. **What Numba is** and when it helps (numeric loops, arrays, no Python objects in hot path)
2. **Basics**: `@njit`, first-run vs cached execution, simple numeric examples
3. **Typing and restrictions**: supported types, unsupported Python features
4. **Sorting methods**: comparison sort in pure Python vs Numba (with timing)
5. **Trading indicators**: SMA, EMA, or a simple momentum-style calculation (Numba-accelerated)
6. **Numerical methods**: linear systems (e.g. Jacobi iteration) and ODEs (e.g. Euler or Runge–Kutta)
7. **Best practices, pitfalls, and when to choose Numba vs Cython**

Study cases use **timing** so you can see the speedup from JIT compilation.

### Contents

- [1. What Numba Is and When It Helps](#1-what-numba-is-and-when-it-helps)
- [2. Typing and Restrictions](#2-typing-and-restrictions)
- [3. Study Case: Sorting Methods](#3-study-case-sorting-methods)
- [4. Study Case: Trading Indicators](#4-study-case-trading-indicators)
- [5. Study Case: Linear Systems (Jacobi Iteration)](#5-study-case-linear-systems-jacobi-iteration)
- [6. Study Case: Ordinary Differential Equations (Euler and Runge–Kutta)](#6-study-case-ordinary-differential-equations-euler-and-rungekutta)
- [7. Best Practices, Pitfalls, and Numba vs Cython](#7-best-practices-pitfalls-and-numba-vs-cython)

## 1. What Numba Is and When It Helps

- **Numba** compiles selected Python functions to native code at runtime (JIT) or ahead of time.
- It targets **numeric code**: loops over scalars and **NumPy arrays**, with types it can infer
  or you declare. It does **not** compile arbitrary Python (no dicts of strings, complex
  objects, etc. in the hot path).
- **`@njit`** (or `@jit(nopython=True)`) means "compile to machine code with no Python
  fallback"; first call pays compilation cost, later calls are fast.
- Best for: **tight loops**, **array-oriented** algorithms, **math-heavy** code (indicators,
  solvers, simulations). Often easier than Cython for "drop-in" speedup of NumPy-adjacent code.

In [1]:
# 1.1 Check Numba and a minimal @njit example

import time

try:
    import numba
    print("Numba version:", numba.__version__)
except ImportError:
    print("Install Numba: pip install numba")

Numba version: 0.61.2


In [2]:
# 1.2 First @njit: sum of squares (compile on first call, then fast)

from numba import njit


@njit  # nopython=True by default: compile to machine code, no Python fallback
def sum_squares_numba(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total


# First call: JIT compiles the function, then runs (slow)
t0 = time.perf_counter()
result1 = sum_squares_numba(1_000_000)
first_time = time.perf_counter() - t0

# Second call: uses cached compiled code (fast)
t0 = time.perf_counter()
result2 = sum_squares_numba(1_000_000)
second_time = time.perf_counter() - t0

print("Result:", result1)
print(f"First call (compile + run):  {first_time*1000:.2f} ms")
print(f"Second call (cached):       {second_time*1000:.2f} ms")

Result: 333332833333500000
First call (compile + run):  645.95 ms
Second call (cached):       0.02 ms


In [3]:
# 1.3 Compare with pure Python (same loop)

def sum_squares_py(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total


# Warm-up numba (already done above)
_ = sum_squares_numba(100)

n = 2_000_000
t0 = time.perf_counter()
sum_squares_py(n)
py_time = time.perf_counter() - t0
t0 = time.perf_counter()
sum_squares_numba(n)
nb_time = time.perf_counter() - t0
print(f"n = {n}: Python {py_time*1000:.2f} ms  vs  Numba {nb_time*1000:.2f} ms  -> speedup {py_time/nb_time:.1f}x")

n = 2000000: Python 66.66 ms  vs  Numba 0.03 ms  -> speedup 2424.0x


## 2. Typing and Restrictions

Numba's **nopython** mode supports:

- **Scalars**: int, float, complex, bool
- **NumPy arrays** (dtypes it knows)
- **Tuples** of supported types
- **Control flow**: if/else, for, while, break, continue
- **No**: arbitrary Python objects, list of strings, dict with string keys in hot path,
  exceptions (in nopython), or calls into non-Numba Python code inside the compiled function

When in doubt, keep the **hot loop** purely numeric and call Numba from Python for I/O and
complex logic.

## 3. Study Case: Sorting Methods

Classic **comparison-based sorting** (e.g. insertion sort or a simple quicksort) in a tight
loop is a good fit for Numba: lots of integer/float comparisons and array indexing.

We'll implement a **simple insertion sort** (or bubble sort) on a NumPy array in pure Python
and with `@njit`, then compare timings. For large arrays you'd use NumPy's built-in sort;
this illustrates the pattern for custom comparison logic that Numba can accelerate.

In [4]:
# 3.1 Insertion sort: pure Python vs Numba

import numpy as np
from numba import njit


def insertion_sort_py(arr: np.ndarray) -> None:
    """In-place insertion sort (pure Python, works on NumPy array)."""
    n = len(arr)
    for i in range(1, n):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key


@njit
def insertion_sort_numba(arr: np.ndarray) -> None:
    """Same algorithm, Numba-compiled."""
    n = len(arr)
    for i in range(1, n):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key


# Small sanity check
a = np.array([3.0, 1.0, 4.0, 1.5, 2.0])
insertion_sort_numba(a)
print("After Numba insertion sort:", a)

After Numba insertion sort: [1.  1.5 2.  3.  4. ]


In [5]:
# 3.2 Timing: insertion sort Python vs Numba

np.random.seed(42)
n = 5000  # insertion sort is O(n^2), so we keep n moderate
repeats = 5

# Warm-up Numba
_ = insertion_sort_numba(np.array([1.0, 2.0]))

py_times = []
nb_times = []
for _ in range(repeats):
    arr_py = np.random.randn(n).astype(np.float64)
    arr_nb = arr_py.copy()
    t0 = time.perf_counter()
    insertion_sort_py(arr_py)
    py_times.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    insertion_sort_numba(arr_nb)
    nb_times.append(time.perf_counter() - t0)

print(f"n = {n}, repeats = {repeats}")
print(f"Python:  {np.mean(py_times)*1000:.2f} ms (mean)")
print(f"Numba:  {np.mean(nb_times)*1000:.2f} ms (mean)")
print(f"Speedup: {np.mean(py_times)/np.mean(nb_times):.1f}x")

n = 5000, repeats = 5
Python:  833.18 ms (mean)
Numba:  2.79 ms (mean)
Speedup: 298.8x


## 4. Study Case: Trading Indicators

Many indicators are **rolling window** or **recurrence** calculations over price arrays:
SMA (simple moving average), EMA (exponential moving average), and custom formulas.
These are ideal for Numba: tight loops over floats and array indexing.

We'll implement:

- **SMA**: average of the last `w` prices.
- **EMA**: exponential moving average with smoothing factor `alpha = 2 / (1 + period)`.

Both in pure Python and with `@njit`, then compare timings on a long price series.

In [6]:
# 4.1 SMA and EMA: pure Python and Numba

@njit
def sma_numba(prices: np.ndarray, window: int) -> np.ndarray:
    """Simple moving average; first (window-1) values are zero."""
    n = len(prices)
    out = np.zeros(n)
    for i in range(window - 1, n):
        s = 0.0
        for j in range(window):
            s += prices[i - j]
        out[i] = s / window
    return out


@njit
def ema_numba(prices: np.ndarray, period: int) -> np.ndarray:
    """Exponential moving average; alpha = 2 / (1 + period)."""
    n = len(prices)
    out = np.zeros(n)
    alpha = 2.0 / (1.0 + period)
    out[0] = prices[0]
    for i in range(1, n):
        out[i] = alpha * prices[i] + (1.0 - alpha) * out[i - 1]
    return out


def sma_py(prices: np.ndarray, window: int) -> np.ndarray:
    n = len(prices)
    out = np.zeros(n)
    for i in range(window - 1, n):
        out[i] = np.mean(prices[i - window + 1 : i + 1])
    return out


def ema_py(prices: np.ndarray, period: int) -> np.ndarray:
    n = len(prices)
    out = np.zeros(n)
    alpha = 2.0 / (1.0 + period)
    out[0] = float(prices[0])
    for i in range(1, n):
        out[i] = alpha * prices[i] + (1.0 - alpha) * out[i - 1]
    return out


# Sanity check
p = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], dtype=np.float64)
print("SMA(3):", sma_numba(p, 3))
print("EMA(3):", ema_numba(p, 3))

SMA(3): [0. 0. 2. 3. 4. 5. 6. 7.]
EMA(3): [1.        1.5       2.25      3.125     4.0625    5.03125   6.015625
 7.0078125]


In [7]:
# 4.2 Timing: indicators on a long price series

n_prices = 100_000
window = 20
prices = np.cumsum(np.random.randn(n_prices).astype(np.float64))  # random walk

# Warm-up
_ = sma_numba(prices[:1000], window)
_ = ema_numba(prices[:1000], window)

for name, func_py, func_nb, args in [
    ("SMA", sma_py, sma_numba, (prices, window)),
    ("EMA", ema_py, ema_numba, (prices, window)),
]:
    t0 = time.perf_counter()
    func_py(*args)
    py_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    func_nb(*args)
    nb_t = time.perf_counter() - t0
    print(f"{name}: Python {py_t*1000:.2f} ms  Numba {nb_t*1000:.2f} ms  speedup {py_t/nb_t:.1f}x")

SMA: Python 228.37 ms  Numba 0.89 ms  speedup 255.6x
EMA: Python 19.43 ms  Numba 0.18 ms  speedup 110.3x


## 5. Study Case: Linear Systems (Jacobi Iteration)

Solving **Ax = b** iteratively with the **Jacobi method**: repeatedly update
`x_new[i] = (b[i] - sum(A[i,j]*x[j] for j != i)) / A[i,i]` until convergence.

This is a double loop over the matrix and vector — a natural fit for Numba.
We'll implement it in pure Python and with `@njit`, then time a few iterations.

In [8]:
# 5.1 Jacobi iteration: Python vs Numba

@njit
def jacobi_step_numba(A: np.ndarray, b: np.ndarray, x: np.ndarray) -> np.ndarray:
    """One Jacobi step: x_new[i] = (b[i] - sum(A[i,j]*x[j] for j!=i)) / A[i,i]."""
    n = len(b)
    x_new = np.empty(n)
    for i in range(n):
        sigma = 0.0
        for j in range(n):
            if j != i:
                sigma += A[i, j] * x[j]
        x_new[i] = (b[i] - sigma) / A[i, i]
    return x_new


@njit
def jacobi_numba(A: np.ndarray, b: np.ndarray, x0: np.ndarray, max_iter: int) -> np.ndarray:
    x = x0.copy()
    for _ in range(max_iter):
        x = jacobi_step_numba(A, b, x)
    return x


def jacobi_step_py(A: np.ndarray, b: np.ndarray, x: np.ndarray) -> np.ndarray:
    n = len(b)
    x_new = np.empty(n)
    for i in range(n):
        sigma = np.dot(A[i, :], x) - A[i, i] * x[i]
        x_new[i] = (b[i] - sigma) / A[i, i]
    return x_new


def jacobi_py(A: np.ndarray, b: np.ndarray, x0: np.ndarray, max_iter: int) -> np.ndarray:
    x = x0.copy()
    for _ in range(max_iter):
        x = jacobi_step_py(A, b, x)
    return x


# Small test: 3x3 system
A = np.array([[4.0, -1.0, 0.0], [-1.0, 3.0, -1.0], [0.0, -1.0, 2.0]])
b = np.array([15.0, 10.0, 10.0])
x0 = np.zeros(3)
x_nb = jacobi_numba(A, b, x0, 20)
print("Jacobi (Numba) x:", x_nb)
print("Residual A@x - b:", np.abs(A @ x_nb - b))

Jacobi (Numba) x: [5.83332856 8.33332539 9.16665713]
Residual A@x - b: [1.11262004e-05 9.53674317e-06 1.11262004e-05]


In [9]:
# 5.2 Timing: Jacobi on a larger system

size = 200
np.random.seed(1)
# Diagonally dominant so Jacobi converges
A = np.random.randn(size, size).astype(np.float64) * 0.1 + np.eye(size) * 2.0
b = np.random.randn(size).astype(np.float64)
x0 = np.zeros(size)
max_iter = 100

_ = jacobi_numba(A, b, x0, 2)  # warm-up

t0 = time.perf_counter()
jacobi_py(A, b, x0, max_iter)
py_t = time.perf_counter() - t0
t0 = time.perf_counter()
jacobi_numba(A, b, x0, max_iter)
nb_t = time.perf_counter() - t0
print(f"Jacobi {size}x{size}, {max_iter} iter: Python {py_t*1000:.0f} ms  Numba {nb_t*1000:.0f} ms  speedup {py_t/nb_t:.1f}x")

Jacobi 200x200, 100 iter: Python 19 ms  Numba 2 ms  speedup 10.3x


## 6. Study Case: Ordinary Differential Equations (Euler and Runge–Kutta)

Integrating **dy/dt = f(t, y)** with a fixed step size is another classic numerical loop.
We'll implement:

- **Euler**: `y_{n+1} = y_n + h * f(t_n, y_n)`
- **RK2 (midpoint)**: `k1 = f(t_n, y_n)`, `k2 = f(t_n + h/2, y_n + h*k1/2)`, `y_{n+1} = y_n + h*k2`

Example: `y' = -y`, `y(0)=1` (solution `y = exp(-t)`). We'll run many steps and compare
Python vs Numba timing.

In [10]:
# 6.1 Euler and RK2 integrators: Python vs Numba

# RHS: dy/dt = -y (solution y = exp(-t))
@njit
def f_numba(t: float, y: float) -> float:
    return -y


@njit
def euler_numba(f, y0: float, t0: float, t_end: float, n_steps: int) -> np.ndarray:
    """Euler method; returns array of y values (including y0)."""
    h = (t_end - t0) / n_steps
    ys = np.empty(n_steps + 1)
    ys[0] = y0
    t = t0
    for i in range(n_steps):
        ys[i + 1] = ys[i] + h * f(t, ys[i])
        t += h
    return ys


@njit
def rk2_numba(f, y0: float, t0: float, t_end: float, n_steps: int) -> np.ndarray:
    """RK2 (midpoint) method."""
    h = (t_end - t0) / n_steps
    ys = np.empty(n_steps + 1)
    ys[0] = y0
    t = t0
    for i in range(n_steps):
        k1 = f(t, ys[i])
        k2 = f(t + 0.5 * h, ys[i] + 0.5 * h * k1)
        ys[i + 1] = ys[i] + h * k2
        t += h
    return ys


def f_py(t: float, y: float) -> float:
    return -y


def euler_py(f, y0: float, t0: float, t_end: float, n_steps: int) -> np.ndarray:
    h = (t_end - t0) / n_steps
    ys = np.empty(n_steps + 1)
    ys[0] = y0
    t = t0
    for i in range(n_steps):
        ys[i + 1] = ys[i] + h * f(t, ys[i])
        t += h
    return ys


def rk2_py(f, y0: float, t0: float, t_end: float, n_steps: int) -> np.ndarray:
    h = (t_end - t0) / n_steps
    ys = np.empty(n_steps + 1)
    ys[0] = y0
    t = t0
    for i in range(n_steps):
        k1 = f(t, ys[i])
        k2 = f(t + 0.5 * h, ys[i] + 0.5 * h * k1)
        ys[i + 1] = ys[i] + h * k2
        t += h
    return ys


# Sanity: y(1) ≈ exp(-1)
n_steps = 10_000
y_euler = euler_numba(f_numba, 1.0, 0.0, 1.0, n_steps)
print("Euler y(1):", y_euler[-1], "  exp(-1):", np.exp(-1.0))

Euler y(1): 0.36786104643292905   exp(-1): 0.36787944117144233


In [11]:
# 6.2 Timing: Euler and RK2, many steps

n_steps = 500_000
_ = euler_numba(f_numba, 1.0, 0.0, 1.0, 1000)  # warm-up
_ = rk2_numba(f_numba, 1.0, 0.0, 1.0, 1000)

for name, func_py, func_nb in [
    ("Euler", lambda: euler_py(f_py, 1.0, 0.0, 1.0, n_steps), lambda: euler_numba(f_numba, 1.0, 0.0, 1.0, n_steps)),
    ("RK2",   lambda: rk2_py(f_py, 1.0, 0.0, 1.0, n_steps),   lambda: rk2_numba(f_numba, 1.0, 0.0, 1.0, n_steps)),
]:
    t0 = time.perf_counter()
    func_py()
    py_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    func_nb()
    nb_t = time.perf_counter() - t0
    print(f"{name}: Python {py_t*1000:.0f} ms  Numba {nb_t*1000:.0f} ms  speedup {py_t/nb_t:.1f}x")

Euler: Python 131 ms  Numba 1 ms  speedup 103.3x
RK2: Python 178 ms  Numba 2 ms  speedup 98.4x


## 7. Best Practices, Pitfalls, and Numba vs Cython

### Best practices

- **Warm-up**: first call to a `@njit` function compiles; time the second call or run a
  small warm-up before benchmarking.
- **Keep hot path nopython**: avoid Python objects, list/dict of strings, and calls into
  non-Numba code inside the compiled function.
- **Use NumPy arrays** with concrete dtypes (e.g. `float64`, `int32`); Numba generates
  efficient code for array indexing and slicing.
- **Cache**: `@njit(cache=True)` writes compiled artifacts to disk so the next process
  run skips compilation (useful in production).

### Common pitfalls

- **First-run slowdown**: don't report "Numba is slow" from a single cold run.
- **Object mode**: if Numba falls back to object mode (no `nopython=True`), you get little
  speedup; fix types so nopython succeeds.
- **Large compilation time**: very large or generic functions can compile slowly; split
  into smaller, typed kernels.

### Numba vs Cython (when to use which)

| | Numba | Cython |
|---|--------|--------|
| **Ease of use** | Add `@njit`; often no code change | Write `.pyx`, add types, build |
| **Best for** | NumPy-heavy loops, array code | Mixed Python/C, custom types, libraries |
| **Compilation** | JIT at runtime (or AOT) | Compile ahead of time |
| **Dependencies** | LLVM (bundled with Numba) | C compiler, Cython |

For **numeric loops and array-heavy algorithms** (indicators, solvers, simulations),
Numba is often the fastest path to speed. For **whole-module** control or **calling C libs**,
Cython can be better. Both can coexist in a project.

### Summary

We used **sorting** (insertion sort), **trading indicators** (SMA, EMA), **linear algebra**
(Jacobi iteration), and **ODEs** (Euler, RK2) as study cases. In each, the same algorithm
in **pure Python** vs **Numba** showed substantial speedups (often 10–100× or more) once
the hot loop is nopython-compatible. For trading and quant code, Numba is a practical
tool to accelerate indicators and numerical kernels without leaving Python.